# Example 3 — LTE hybrid planning with ordered replacements

Requires SDK 0.1.34+ and a saved example 2 result. The source remains unchanged.
Run from the workspace notebooks folder. Review extension.toml before preparing.
This example adds Wi-SUN mesh. Its rules explicitly mark good LTE gateways connected and copy their source rank; mesh-only devices start unconnected.


In [ ]:
from pathlib import Path
from cisei_planning_sdk import PlanningClient, PlanningProject, ExtendedPlanningProject, plot_network

client = PlanningClient("http://network-service:8080", workspace="..", timeout=1900.0)
source = PlanningProject("../projects/example_2_LTE_simulation", result_file="result.json")
# If your intended baseline is the alternative result, use result_file="result_link_margin.json".
project = ExtendedPlanningProject("../projects/example_3_LTE_hybrid", source=source)


## 1. Prepare the derived project once

The first rule upgrades lte_leaf devices with rank <= 2; the second replaces the
remaining lte_leaf devices with mesh routers. The report records all assignments.
Repeating preparation reuses an unchanged study. After editing rules, call
project.prepare(result=source.load_result(), overwrite=True) to regenerate it
in place. Extracted features are retained; previous generated inputs and stale
results/views are backed up. No geographic extraction occurs during preparation.


In [ ]:
report = project.prepare(result=source.load_result(), overwrite=True)
from collections import Counter
print(Counter(row["new_profile"] for row in report["replacements"]))
print("Replacement report:", project.path / "replacement_report.json")


## 2. Rebuild LTE and mesh candidates

LTE sector rules, mesh neighbor rules and internal device edges are explicit in the recipe.


In [ ]:
scenario = client.open_project(project, planner="graph")
candidates = scenario.build_candidate_edges()
plot_network(scenario.export_result(), kind="candidate", save_path=project.path / "candidate_network.html");


## 3. Compute link metrics in checkpointed batches

LTE cache hits are reused. New Wi-SUN requests use 915 MHz and must be extracted.
The range, neighbor limit and radio assumptions are examples, not guarantees.


In [ ]:
import os

metrics = scenario.compute_metrics(
    geo_base_url=os.getenv("PLANNING_GEO_URL", "http://features:8080"),
    geo_timeout=300.0,
    batch_size=20,
    on_batch=lambda p: print(f"Saved: {p['next_offset']}/{p['total_links']} links"),
)
scenario.show_metrics()


## 4. Solve and compare the derived study

The combined threshold below is an illustrative accumulated cost cutoff, not the
same quality test as example 2's direct-link cutoff 2. Review individual mesh
links and remaining unserved clients. No automatic pruning is performed.


In [ ]:
solution = scenario.solve()
combined_rank_threshold = 4.0
scenario.show_evaluation(rank_threshold=combined_rank_threshold)
scenario.save_result(rank_threshold=combined_rank_threshold)
saved_result = scenario.load_result()
plot_network(saved_result, kind="final", save_path=project.path / "solution_network.html");
